# 🕌 Athar - Google Colab Environment Setup

**Purpose:** One-click setup for GPU-accelerated embedding and data processing

**What this does:**
- ✅ Installs all required dependencies
- ✅ Configures GPU support
- ✅ Sets up project structure
- ✅ Validates installation
- ✅ Tests GPU availability

**Run Time:** ~5 minutes

---

## 1️⃣ Check GPU Availability

In [ ]:
import torch

print("="*60)
print("🕌 ATHAR - GPU SETUP CHECK")
print("="*60)

if torch.cuda.is_available():
    print(f"\n✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   GPU Count: {torch.cuda.device_count()}")
else:
    print("\n⚠️  No GPU detected. Running on CPU (will be slower)")
    print("   Go to: Runtime → Change runtime type → T4 GPU")

print(f"\n📊 System Info:")
print(f"   PyTorch: {torch.__version__}")
print(f"   Python: {torch.version}")

## 2️⃣ Install Dependencies

In [ ]:
%%bash
echo "📦 Installing dependencies..."

# Core ML libraries
pip install -q transformers torch accelerate

# Vector database
pip install -q qdrant-client

# Utilities
pip install -q numpy pandas tqdm

# HuggingFace authentication (if needed)
pip install -q huggingface_hub

echo "✅ Dependencies installed successfully"

## 3️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import os

print("📂 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Set up Athar directory
ATHAR_DIR = "/content/drive/MyDrive/Athar"
os.makedirs(ATHAR_DIR, exist_ok=True)

print(f"\n✅ Google Drive mounted at: {ATHAR_DIR}")
print(f"\n📁 Directory structure:")
print(f"   {ATHAR_DIR}/")
print(f"   ├── datasets/         # Upload datasets here")
print(f"   ├── output/           # Generated embeddings")
print(f"   └── notebooks/        # This notebook")

## 4️⃣ Clone Athar Repository

In [ ]:
import os

ATHAR_REPO = "/content/athar"

if not os.path.exists(ATHAR_REPO):
    print("📥 Cloning Athar repository...")
    !git clone https://github.com/Kandil7/Athar.git {ATHAR_REPO}
    print(f"\n✅ Repository cloned at: {ATHAR_REPO}")
else:
    print("✅ Repository already exists")
    print("📂 Pulling latest changes...")
    %cd {ATHAR_REPO}
    !git pull

## 5️⃣ Set Up HuggingFace Token

In [ ]:
from huggingface_hub import login
from google.colab import userdata

print("🔑 Setting up HuggingFace token...")

# Option 1: From Colab secrets (recommended)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("✅ Logged in using Colab secret")
    else:
        print("⚠️  No HF_TOKEN in secrets. Enter manually below:")
        HF_TOKEN = input("Enter HuggingFace token: ")
        login(token=HF_TOKEN)
        print("✅ Logged in successfully")
except Exception as e:
    print(f"⚠️  Error: {e}")
    print("   You can still run notebooks that don't require HF authentication")

print("\n💡 To save token for future sessions:")
print("   1. Go to Colab sidebar → Secrets")
print("   2. Add HF_TOKEN with your token value")
print("   3. Enable 'Notebook access'")

## 6️⃣ Test Embedding Model

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

print("🧪 Testing embedding model...")

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n📥 Loading model: {MODEL_NAME}")
print(f"   Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to(device)

# Test encoding
test_text = "بسم الله الرحمن الرحيم"
inputs = tokenizer(test_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:, 0]

print(f"\n✅ Model loaded successfully!")
print(f"   Embedding dimension: {embedding.shape[1]}")
print(f"   Test text: {test_text}")
print(f"   Embedding shape: {embedding.shape}")

# Clean up
del model, tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n🎉 Setup complete! You can now run the embedding notebooks.")

## 📋 Next Steps

After setup is complete, run one of the main notebooks:

1. **01_embed_all_collections.ipynb** - Embed all 10 collections
2. **02_process_sanadset_hadith.ipynb** - Process 650K hadith
3. **03_analyze_and_chunk_books.ipynb** - Analyze and chunk books

**Estimated Time Savings with GPU:**
- CPU embedding: ~900 hours for 650K hadith
- GPU (T4) embedding: ~6 hours for 650K hadith
- **Savings: 150x faster!** 🚀